# 08 — Baselines clássicos (TDAH): SVM, Random Forest e XGBoost (protocolo revisado)

**O que mudou em relação à versão anterior**

| Antes | Agora | Motivo |
|---|---|---|
| Hiperparâmetros fixos (sem busca) | Busca de hiperparâmetros aninhada, métrica de seleção **AUC-ROC** | AUC é a métrica principal do TCC; o F1 é instável com folds pequenos |
| Folds recriados no notebook | **Mesmos folds externos do notebook 07**, só lidos e auditados | Todos os modelos da base precisam compartilhar os folds |
| Sem baseline trivial | `DummyClassifier` + teste com rótulos embaralhados | Mostrar o que é "acaso" nesta base |
| Resultados só em tabela | Tabela longa por base/modelo/fold/semente + previsões OOF salvas | Rastreabilidade para o notebook 12 |

**Protocolo**
- 5 folds externos agrupados por site (do notebook 07); cada fold inteiro vira teste uma vez.
- Busca interna: até 3 folds agrupados por site, **somente se cada divisão tiver as duas classes** (checado antes, célula 3). Se 3 não for viável, tenta 2; se nenhuma for, usa hiperparâmetros padrão **e registra isso** — nunca cria split aleatório.
- Grades: SVM (C × gamma), Random Forest (árvores × profundidade × max_features), XGBoost (estimadores × profundidade × learning rate), exatamente como no plano. `scale_pos_weight` e `StandardScaler` usam só o treino de cada divisão.
- Sete métricas por fold. Os folds são validação cruzada por site, **não** um conjunto de teste independente.

In [1]:
# 1. Projeto, módulo de avaliação e dataset (TDAH / ADHD-200)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
_raiz = next((p for p in [_cwd, *_cwd.parents] if (p / 'config.py').is_file()), None)
if _raiz is None:
    raise FileNotFoundError('Não encontrei config.py. Abra o notebook dentro de TCC_TDAH/notebooks.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import config
import avaliacao as av          # avaliacao.py fica ao lado do config.py
config.criar_pastas()

SEED = int(getattr(config, 'SEED', 42))
NOME_BASE = 'TDAH'
ROTULO_CASO = 'TDAH'
ARQ_DATASET = Path(config.ARQ_DATASET)
PASTA_RES = Path(getattr(config, 'RESULTS', _raiz / 'results'))
PASTA_MOD = Path(getattr(config, 'MODELS', _raiz / 'models'))
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros desta execução
GRADE_REDUZIDA = False       # True só se a grade completa estourar o tempo (documentar no TCC)
SEMENTES = [SEED]            # a busca interna e o treino são determinísticos dado a semente
N_PERMUTACOES = 3            # teste de rótulos embaralhados (por fold)
EXCLUIR_SITES = []           # sensibilidade: p.ex. ['Pittsburgh', 'WashU'] no TDAH (ver nota na célula 3)
MODELOS = ['SVM_RBF', 'Random_Forest', 'XGBoost']
if av.RAPIDO:
    N_PERMUTACOES = 1
    print('*** TCC_RAPIDO=1: grades e permutações mínimas (só para testar o notebook) ***')

d = av.carregar_dataset(ARQ_DATASET)
d = av.filtrar_sites(d, EXCLUIR_SITES)
SUFIXO = '' if not EXCLUIR_SITES else '_sem_' + '_'.join(s.lower() for s in EXCLUIR_SITES)

print(f'Base: {NOME_BASE} | X: {d["X"].shape} | ROIs: {av.n_rois_de(d["X"].shape[1])}')
print('Classes (0=controle, 1=%s):' % ROTULO_CASO, pd.Series(d['y']).value_counts().sort_index().to_dict())
print('Sites:', len(np.unique(d['sites'])), '| Folds externos:', sorted(np.unique(d['fold_id']).tolist()))
print('Intervalo de X (Fisher z esperado, não [-1,1]):', float(d['X'].min()), 'a', float(d['X'].max()))

Base: TDAH | X: (346, 17955) | ROIs: 190
Classes (0=controle, 1=TDAH): {0: 185, 1: 161}
Sites: 4 | Folds externos: [0, 1, 2, 3]
Intervalo de X (Fisher z esperado, não [-1,1]): -1.5912327766418457 a 1.65999436378479


**Análise de sensibilidade (TDAH):** Pittsburgh e WashU só têm controles, então site e diagnóstico ficam confundidos neles. Para a análise secundária, rode este notebook de novo com `EXCLUIR_SITES = ['Pittsburgh', 'WashU']` (use os nomes exatamente como aparecem na tabela de sites). Os folds dos demais sites são os mesmos; os arquivos saem com sufixo `_sem_...` e **não** se misturam com a análise principal. A contagem esperada (~574 = 256 TDAH + 318 controles) está **pendente de verificação**: confira a linha impressa por `filtrar_sites`.

In [2]:
# 3. Auditoria dos folds externos e viabilidade dos splits internos
print('FOLDS EXTERNOS')
display(av.resumo_folds(d))

print('\nSITE x CLASSE (sites com uma só classe aparecem com 0 em uma coluna)')
display(pd.crosstab(pd.Series(d['sites'], name='site'), pd.Series(d['y'], name='classe')))

print('\nVIABILIDADE DA BUSCA INTERNA AGRUPADA POR SITE (por fold externo)')
aud_int = av.auditoria_splits_internos(d, k_max=3)
display(aud_int)
if not aud_int['viavel'].all():
    print('ATENÇÃO: há fold externo sem partição interna viável; ali serão usados hiperparâmetros padrão (registrado).')
if (aud_int['k_usado'] < 3).any():
    print('Há fold(s) com menos de 3 folds internos; isso fica registrado na coluna k_interno dos resultados.')

FOLDS EXTERNOS


,fold,n_treino,n_teste,controles_teste,casos_teste,controles_treino,casos_treino,sites_teste,sobreposicao_sites,duas_classes_no_teste
0,0,266,80,58,22,127,139,KKI,0,True
1,1,156,190,87,103,98,58,NYU,0,True
2,2,311,35,19,16,166,145,OHSU,0,True
3,3,305,41,21,20,164,141,NeuroIMAGE,0,True



SITE x CLASSE (sites com uma só classe aparecem com 0 em uma coluna)


classe,0,1
site,,
KKI,58,22
NYU,87,103
NeuroIMAGE,21,20
OHSU,19,16



VIABILIDADE DA BUSCA INTERNA AGRUPADA POR SITE (por fold externo)


,fold_externo,k_pedido,n_sites_treino,k_usado,semente_particao,viavel,tam_validacoes
0,0,3,3,3,42,True,"[190, 35, 41]"
1,1,3,3,3,42,True,"[80, 35, 41]"
2,2,3,3,3,42,True,"[80, 41, 190]"
3,3,3,3,3,42,True,"[80, 190, 35]"


In [3]:
# 4. Treino e avaliação (nested: busca interna -> refit no treino externo -> teste no fold externo)
from sklearn.dummy import DummyClassifier
from sklearn.base import clone

folds = sorted(np.unique(d['fold_id']).tolist())
nomes_todos = ['Dummy_prior', 'Dummy_estratificado'] + MODELOS
proba_oof = {m: np.full(len(d['y']), np.nan, dtype=np.float32) for m in nomes_todos}
pred_oof = {m: np.full(len(d['y']), -1, dtype=np.int8) for m in nomes_todos}
linhas, tabelas_busca = [], []

for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]
    tr = np.where(d['fold_id'] != fold)[0]
    Xtr, ytr, Xte, yte = d['X'][tr], d['y'][tr], d['X'][te], d['y'][te]
    splits, info = av.splits_internos(ytr, d['sites'][tr], k_max=3, seed=SEED)
    print(f'\n=== Fold {fold}: treino={len(tr)} teste={len(te)} | folds internos: ' +
          (f"{info['k_usado']}" if info['viavel'] else 'inviável -> hiperparâmetros padrão'))

    # Dummies (referência de acaso)
    for nome, estrategia in [('Dummy_prior', 'prior'), ('Dummy_estratificado', 'stratified')]:
        dm = DummyClassifier(strategy=estrategia, random_state=SEED).fit(Xtr, ytr)
        p = dm.predict_proba(Xte)[:, 1]
        pr = dm.predict(Xte)
        proba_oof[nome][te], pred_oof[nome][te] = p, pr
        linhas.append(av.linha_resultado(NOME_BASE, nome, fold, SEED, yte, pr, p, k_interno=0, params='-'))

    for nome in MODELOS:
        for sem in SEMENTES:
            t0 = av.agora()
            if info['viavel']:
                melhores, tab = av.busca_hiperparametros(nome, Xtr, ytr, splits, seed=sem, reduzida=GRADE_REDUZIDA)
                tab.insert(0, 'fold_externo', fold); tab.insert(0, 'modelo', nome); tab.insert(0, 'semente', sem)
                tabelas_busca.append(tab)
            else:
                melhores = {}
            t_busca = av.agora() - t0
            modelo = av.criar_modelo(nome, ytr, seed=sem, params=melhores)
            modelo.fit(Xtr, ytr)               # refit com TODO o treino externo
            p = av.pontuar(modelo, Xte)
            pr = modelo.predict(Xte).astype(np.int8)
            if sem == SEMENTES[0]:
                proba_oof[nome][te], pred_oof[nome][te] = p, pr
            lin = av.linha_resultado(NOME_BASE, nome, fold, sem, yte, pr, p,
                                     k_interno=int(info['k_usado']), params=str(melhores),
                                     seg_busca=round(t_busca, 1), seg_total=round(av.agora() - t0, 1))
            linhas.append(lin)
            aviso = '  <-- ALERTA: previu uma única classe' if lin['classe_unica_prevista'] else ''
            print(f"  {nome:14s} sem={sem} AUC={lin['auc_roc']:.3f} BalAcc={lin['balanced_accuracy']:.3f} "
                  f"Rec={lin['recall']:.2f} Esp={lin['specificity']:.2f} | {t_busca/60:.1f} min de busca | {melhores}{aviso}")
    pd.DataFrame(linhas).to_csv(PASTA_RES / f'08_resultados_long{SUFIXO}.csv', index=False)   # checkpoint por fold

for m in nomes_todos:
    if np.any(pred_oof[m] < 0) or not np.isfinite(proba_oof[m]).all():
        raise RuntimeError(f'Previsões OOF incompletas para {m}.')
df_long = pd.DataFrame(linhas)
df_long['tipo'] = 'externo'
if tabelas_busca:
    pd.concat(tabelas_busca).to_csv(PASTA_RES / f'08_busca_hiperparametros{SUFIXO}.csv', index=False)


=== Fold 0: treino=266 teste=80 | folds internos: 3
  SVM_RBF        sem=42 AUC=0.437 BalAcc=0.500 Rec=1.00 Esp=0.00 | 0.8 min de busca | {'clf__C': 1, 'clf__gamma': 0.001}  <-- ALERTA: previu uma única classe
  Random_Forest  sem=42 AUC=0.507 BalAcc=0.527 Rec=0.73 Esp=0.33 | 1.1 min de busca | {'n_estimators': 500, 'max_depth': None, 'max_features': 'log2'}
  XGBoost        sem=42 AUC=0.493 BalAcc=0.511 Rec=0.59 Esp=0.43 | 32.8 min de busca | {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01}

=== Fold 1: treino=156 teste=190 | folds internos: 3
  SVM_RBF        sem=42 AUC=0.487 BalAcc=0.500 Rec=1.00 Esp=0.00 | 0.3 min de busca | {'clf__C': 0.1, 'clf__gamma': 0.001}  <-- ALERTA: previu uma única classe
  Random_Forest  sem=42 AUC=0.523 BalAcc=0.499 Rec=0.01 Esp=0.99 | 0.8 min de busca | {'n_estimators': 500, 'max_depth': None, 'max_features': 'log2'}
  XGBoost        sem=42 AUC=0.547 BalAcc=0.527 Rec=0.20 Esp=0.85 | 14.3 min de busca | {'n_estimators': 300, 'max_depth': 5, 

In [4]:
# 5. Sanidade: rótulos de treino embaralhados (a AUC deve ficar perto de 0,5)
# Usa o SVM com hiperparâmetros padrão (sem busca) para ser rápido. Se a AUC aqui ficar claramente
# acima de 0,5, há vazamento ou erro de pipeline — investigue antes de interpretar qualquer resultado.
rng = np.random.default_rng(SEED)
linhas_perm = []
for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]
    tr = np.where(d['fold_id'] != fold)[0]
    for k in range(N_PERMUTACOES):
        y_emb = rng.permutation(d['y'][tr])
        m = av.criar_modelo('SVM_RBF', y_emb, seed=SEED).fit(d['X'][tr], y_emb)
        p = av.pontuar(m, d['X'][te])
        linhas_perm.append(dict(fold=fold, perm=k, auc=float(av.roc_auc_score(d['y'][te], p))))
df_perm = pd.DataFrame(linhas_perm)
print('AUC com rótulos embaralhados (SVM):  média = %.3f  | mín = %.3f  | máx = %.3f' %
      (df_perm.auc.mean(), df_perm.auc.min(), df_perm.auc.max()))
print('Esperado: ~0,5 (variação de ±0,05 a ±0,10 é normal em folds pequenos).')
df_perm.to_csv(PASTA_RES / f'08_teste_rotulos_embaralhados{SUFIXO}.csv', index=False)

AUC com rótulos embaralhados (SVM):  média = 0.527  | mín = 0.427  | máx = 0.633
Esperado: ~0,5 (variação de ±0,05 a ±0,10 é normal em folds pequenos).


In [5]:
# 6. Resumo: média por fold, média ponderada pelo tamanho, AUC OOF com IC por bootstrap de sites
res = av.resumir_resultados(df_long, d, proba_oof)
pd.set_option('display.width', 200)
print('MÉDIA ± DP ENTRE FOLDS (1 linha por modelo)')
display(res['por_fold'])
print('\nMÉDIA PONDERADA PELO TAMANHO DO FOLD')
display(res['ponderado'])
print('\nAUC OOF (previsões fora do treino reunidas) com IC 95% por bootstrap de sites')
display(res['auc_oof_ic'])
print('Critério definido a priori: IC inferior > 0,5 => "acima do acaso". Não é prova; com poucos sites o IC é largo.')
print('Não se usa Wilcoxon com 5 folds. Diferenças entre modelos são só reportadas, com sua incerteza.')

print('\nDESEMPENHO POR SITE (modelo com maior AUC OOF)')
melhor = res['auc_oof_ic'].sort_values('auc', ascending=False).iloc[0]['modelo']
tab_site = av.tabela_por_site(d, proba_oof[melhor], pred_oof[melhor])
print('Modelo:', melhor)
display(tab_site.round(3))

MÉDIA ± DP ENTRE FOLDS (1 linha por modelo)


accuracy         balanced_accuracy         precision          recall         specificity              f1         auc_roc        
                        mean     std              mean     std      mean     std    mean     std        mean     std    mean     std    mean     std
modelo                                                                                                                                              
Dummy_estratificado   0.4488  0.0338            0.4555  0.0464    0.3861  0.1164  0.3769  0.1143      0.5341  0.0921  0.3627  0.0438  0.4555  0.0464
Dummy_prior           0.4470  0.1199            0.5000  0.0000    0.0688  0.1375  0.2500  0.5000      0.7500  0.5000  0.1078  0.2157  0.5000  0.0000
Random_Forest         0.4744  0.0337            0.5015  0.0210    0.3131  0.2277  0.2780  0.3467      0.7250  0.3097  0.2121  0.2341  0.5175  0.0087
SVM_RBF               0.4680  0.1295            0.5000  0.0000    0.2043  0.2599  0.5000  0.5774      0.5000  0.5774  0.2836  0.3458  0.5029  0.0878
XGBoost               0.5136  0.0319            0.5260  0.0117    0.5167  0.1710  0.3487  0.2341      0.7032  0.2404  0.3407  0.1367  0.5229  0.1094


MÉDIA PONDERADA PELO TAMANHO DO FOLD


,modelo,accuracy_pond,balanced_accuracy_pond,precision_pond,recall_pond,specificity_pond,f1_pond,auc_roc_pond
0,Dummy_estratificado,0.4682,0.4800,0.4417,0.3852,0.5747,0.3900,0.4800
1,Dummy_prior,0.4306,0.5000,0.0636,0.2312,0.7688,0.0997,0.5000
2,Random_Forest,0.4624,0.5034,0.3885,0.2114,0.7953,0.1484,0.5188
3,SVM_RBF,0.4769,0.5000,0.3613,0.7803,0.2197,0.4858,0.4894
4,XGBoost,0.5029,0.5246,0.5341,0.3110,0.7382,0.3279,0.5320



AUC OOF (previsões fora do treino reunidas) com IC 95% por bootstrap de sites


,modelo,auc,ic_inf,ic_sup,n_boot_validos,obs
0,Dummy_prior,NaN,NaN,NaN,0,saída constante por fold: usar a AUC por fold ...
1,Dummy_estratificado,NaN,NaN,NaN,0,saída constante por fold: usar a AUC por fold ...
2,SVM_RBF,0.5648,0.3945,0.6120,2000,
3,Random_Forest,0.4166,0.3805,0.5199,2000,
4,XGBoost,0.4751,0.4496,0.5572,2000,


Critério definido a priori: IC inferior > 0,5 => "acima do acaso". Não é prova; com poucos sites o IC é largo.
Não se usa Wilcoxon com 5 folds. Diferenças entre modelos são só reportadas, com sua incerteza.

DESEMPENHO POR SITE (modelo com maior AUC OOF)
Modelo: SVM_RBF


,site,fold,n,controles,casos,acuracia,auc
0,KKI,0,80,58,22,0.275,0.500
1,NYU,1,190,87,103,0.542,0.500
2,NeuroIMAGE,3,41,21,20,0.512,0.500
3,OHSU,2,35,19,16,0.543,0.457


In [6]:
# 7. Salvar tudo (tabela longa, resumos, OOF)
df_long.to_csv(PASTA_RES / f'08_resultados_long{SUFIXO}.csv', index=False)
res['por_fold'].to_csv(PASTA_RES / f'08_resumo_por_fold{SUFIXO}.csv')
res['ponderado'].to_csv(PASTA_RES / f'08_resumo_ponderado{SUFIXO}.csv', index=False)
res['auc_oof_ic'].to_csv(PASTA_RES / f'08_auc_oof_ic{SUFIXO}.csv', index=False)
tab_site.to_csv(PASTA_RES / f'08_por_site_{melhor}{SUFIXO}.csv', index=False)
av.salvar_oof(PASTA_RES / f'08_predicoes_oof{SUFIXO}.npz', d, proba_oof, pred_oof)
print('Salvo em:', PASTA_RES)
print('Obs.: não é salvo modelo "final" ajustado em todos os dados aqui; para XAI (notebook 11) o modelo é '
      'reajustado por fold com os hiperparâmetros registrados em 08_resultados_long.csv.')

Salvo em: /home/maria/TCC_TDAH/results
Obs.: não é salvo modelo "final" ajustado em todos os dados aqui; para XAI (notebook 11) o modelo é reajustado por fold com os hiperparâmetros registrados em 08_resultados_long.csv.
